# 🏋️ Fine-tune GR00T N1.7 cho task `can_nav_pnp` (SONIC 1.1) trên Google Colab

Robot G1 đi vòng qua vật cản, ngồi xổm nhặt lon trên bục thấp, đứng dậy, bước tới bàn và đặt lon vào khay. Dữ liệu do
`scripts/datagen/can_nav_pnp/generate.py` sinh, xuất bằng `export_gr00t.py`. Model học **cùng action với Ψ₀**: token
SONIC 64 chiều + 2 tay Dex3 (7 + 7), chạy qua SONIC 1.1 khi đánh giá (`closed_loop_eval.py --gr00t HOST:PORT`).

* **Embodiment:** `UNITREE_G1_SONIC` (đã đăng ký sẵn trong Isaac-GR00T, projector 11). Checkpoint gốc
  `nvidia/GR00T-N1.7-3B` chưa có thống kê cho tag này nên thống kê chuẩn hoá lấy từ dữ liệu của ta (`meta/stats.json`).
* **State 46:** chân 6+6, eo 3, tay trái 7, tay phải 7, bàn tay trái 7, bàn tay phải 7, trọng lực chiếu 3.
* **Action 78 × 40 bước (1,33 s ở 30 FPS):** `motion_token` 64, `left_hand_joints` 7, `right_hand_joints` 7, tuyệt đối.
* **Video:** `ego_view` 320×240, 1 khung. **Câu lệnh:** `annotation.human.task_description`.

### Yêu cầu
* **GPU A100** (khoảng 35 GB VRAM với batch 80; GPU 40 GB thì giảm `GLOBAL_BATCH_SIZE` xuống 32 ở Bước 4).
* **HuggingFace:** quyền tải repo gated `nvidia/Cosmos-Reason2-2B`, token trong **Colab Secrets** tên `HF_TOKEN`.
* **Google Drive:** file `MyDrive/gr00t/<DATASET_NAME>_gr00t.zip` (tạo ở máy local, chứa `<DATASET_NAME>_train` và
  `<DATASET_NAME>_val`). Checkpoint chép về `MyDrive/gr00t/checkpoints/`, mỗi cái khoảng 10–15 GB.

### Tạo file zip ở máy local
```bash
.venv-datagen/bin/python scripts/datagen/can_nav_pnp/export_gr00t.py .data/can_nav_160 --out .data/gr00t --zip
```
rồi tải `.data/gr00t/can_nav_160_gr00t.zip` lên thư mục `gr00t` trong Google Drive.


In [ ]:
# [BƯỚC 1] Kiểm tra GPU
!nvidia-smi

In [ ]:
%%bash
# [BƯỚC 2] Cài Isaac-GR00T vào môi trường riêng (.venv, Python 3.12) bằng uv (~5-10 phút).
# set -e: dừng ngay ở lệnh lỗi đầu tiên để thấy rõ nguyên nhân (lệnh "!" của Colab không tự dừng khi lỗi).
set -euxo pipefail
cd /content
[ -d Isaac-GR00T/.git ] || GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/NVIDIA/Isaac-GR00T
cd Isaac-GR00T
git checkout 51d4c89f72fda44cbf77285c6a8114b52676b8a1
# Clone bỏ qua file Git LFS (video, dataset demo) cho nhẹ, nhưng uv cần wheel LFS trong
# scripts/deployment/dgpu/wheels để giải lock (kể cả wheel aarch64 không dùng trên Colab): tải riêng thư mục này.
command -v git-lfs || (apt-get -qq update && apt-get -qq install -y git-lfs)
git lfs install --local
git lfs pull --include="scripts/deployment/dgpu/wheels/*"
# Trình cài uv chọn /usr/local/bin hoặc ~/.local/bin tùy môi trường: gọi uv qua PATH, không dùng đường dẫn cứng
export PATH="$HOME/.local/bin:/usr/local/bin:$PATH"
command -v uv || curl -LsSf https://astral.sh/uv/install.sh | sh
# Venv luôn nằm ở /content/Isaac-GR00T/.venv (bỏ qua biến môi trường nếu có)
unset UV_PROJECT_ENVIRONMENT VIRTUAL_ENV
uv sync --python 3.12
.venv/bin/python -c "import gr00t; print('GR00T installed successfully')"

In [ ]:
# [BƯỚC 3] Nạp HuggingFace token từ Colab Secrets (không dán token trực tiếp vào notebook)
# và kiểm tra ngay quyền tải backbone gated nvidia/Cosmos-Reason2-2B.
# Chạy lại cell này mỗi khi runtime khởi động lại.
import os
from huggingface_hub import hf_hub_download, whoami
from huggingface_hub.utils import GatedRepoError

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception as e:
    print(f"Không đọc được secret HF_TOKEN ({type(e).__name__}); nhập token thủ công.")
    from getpass import getpass
    token = getpass("HuggingFace token: ")
token = (token or "").strip()
assert token, "HF_TOKEN rỗng: thêm secret tên HF_TOKEN (biểu tượng 🔑) và bật Notebook access."
os.environ["HF_TOKEN"] = token

try:
    user = whoami(token=token)
except Exception as e:
    raise RuntimeError(
        "Token không hợp lệ hoặc đã bị thu hồi. Tạo token mới tại https://huggingface.co/settings/tokens "
        "rồi cập nhật secret HF_TOKEN."
    ) from e
print(f"Token hợp lệ, tài khoản: {user['name']}")

try:
    hf_hub_download("nvidia/Cosmos-Reason2-2B", "config.json", token=token)
except GatedRepoError as e:
    raise RuntimeError(
        f"Tài khoản {user['name']} chưa tải được nvidia/Cosmos-Reason2-2B. Kiểm tra:\n"
        "  1. Đã bấm xin quyền và được duyệt tại https://huggingface.co/nvidia/Cosmos-Reason2-2B "
        "(đăng nhập đúng tài khoản trên).\n"
        "  2. Nếu là token fine-grained: bật quyền 'Read access to contents of all public gated repos "
        "you can access', hoặc dùng token loại Read."
    ) from e
print("✅ Có quyền tải nvidia/Cosmos-Reason2-2B.")

In [ ]:
# [BƯỚC 4] Cấu hình. Các biến môi trường ở đây được các cell %%bash phía sau dùng lại.
import os
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/gr00t"
# Tên pack = tham số --name của export_gr00t.py (mặc định tên thư mục dataset gốc). Zip: <DATASET_NAME>_gr00t.zip
DATASET_NAME = "can_nav_160"
os.environ.update({
    "DATASET_ZIP": f"{DRIVE_DIR}/{DATASET_NAME}_gr00t.zip",
    "DATASET_DIR": f"/content/datasets/{DATASET_NAME}_train",
    "TEST_DIR": f"/content/datasets/{DATASET_NAME}_val",
    "OUTPUT_DIR": "/content/outputs",
    "EXPERIMENT": f"{DATASET_NAME}_gr00t",
    "DRIVE_CKPT_DIR": f"{DRIVE_DIR}/checkpoints",
    "EMBODIMENT_TAG": "UNITREE_G1_SONIC",
    # Số bước và batch đặt GIỐNG notebook Ψ₀ để so sánh: batch hiệu dụng 128 (= GLOBAL_BATCH_SIZE x GRAD_ACC) x 8 000 bước
    # = 1,02 triệu mẫu (~3,3 lần đi qua 310 000 khung của can_nav_160). Mọi thứ khác giữ mặc định của Isaac-GR00T
    # (finetune.sh: LR 1e-4 cosine, warmup 5%, bỏ state 20%, không đổi màu ảnh). GPU 40 GB: GLOBAL_BATCH_SIZE 32, GRAD_ACC 4.
    "NUM_GPUS": "1",
    "MAX_STEPS": "8000",
    "SAVE_STEPS": "1000",
    "GLOBAL_BATCH_SIZE": "64",
    "GRAD_ACC": "2",
    "SYNC_FROM_STEP": "2000",   # Bước 8a chép checkpoint từ bước này về Drive
    "DATALOADER_NUM_WORKERS": "4",
    "USE_WANDB": "0",
    "BASE_MODEL_PATH": "nvidia/GR00T-N1.7-3B",
})
assert os.path.exists(os.environ["DATASET_ZIP"]), f"Không thấy {os.environ['DATASET_ZIP']} trên Drive."
print(f"Checkpoint lưu vào: {os.environ['OUTPUT_DIR']}/{os.environ['EXPERIMENT']} -> "
      f"{os.environ['DRIVE_CKPT_DIR']}/{os.environ['EXPERIMENT']}")
print(f"Batch {os.environ['GLOBAL_BATCH_SIZE']} x {os.environ['MAX_STEPS']} bước")
print("✅ Đã mount Drive và đặt cấu hình.")


In [ ]:
%%bash
# [BƯỚC 5] Giải nén dataset từ Drive (zip chứa <DATASET_NAME>_train và <DATASET_NAME>_val)
set -euo pipefail
mkdir -p /content/datasets
[ -d "$DATASET_DIR" ] || unzip -q "$DATASET_ZIP" -d /content/datasets
ls "$DATASET_DIR" "$DATASET_DIR/meta" "$TEST_DIR"


In [ ]:
%%bash
# [BƯỚC 6] Nạp thử dataset bằng data loader của Isaac-GR00T với cấu hình modality unitree_g1_sonic, để phát hiện lỗi
# dataset trước khi nạp model 3B. meta/stats.json do export_gr00t.py ghi sẵn (action tuyệt đối: không cần
# relative_stats.json).
set -euo pipefail
cd /content/Isaac-GR00T
.venv/bin/python - "$DATASET_DIR" <<'EOF'
import json, sys
import numpy as np
from gr00t.configs.data.embodiment_configs import MODALITY_CONFIGS
from gr00t.data.dataset.lerobot_episode_loader import LeRobotEpisodeLoader

path = sys.argv[1]
cfg = MODALITY_CONFIGS["unitree_g1_sonic"]
loader = LeRobotEpisodeLoader(path, cfg)
lengths = loader.get_episode_lengths()
print(f"{len(lengths)} episode, {sum(lengths)} khung, dài {min(lengths)}-{max(lengths)} khung")
df = loader[0]
want = {"state.left_leg": 6, "state.right_leg": 6, "state.waist": 3, "state.left_arm": 7, "state.right_arm": 7,
        "state.left_hand": 7, "state.right_hand": 7, "state.projected_gravity": 3,
        "action.motion_token": 64, "action.left_hand_joints": 7, "action.right_hand_joints": 7}
for k, n in want.items():
    v = np.asarray(df[k].iloc[0])
    assert v.shape[-1] == n, f"{k}: {v.shape} (cần {n})"
    print(f"  {k:28s} {v.shape}")
img = df["video.ego_view"].iloc[0]
print("  video.ego_view", np.asarray(img).shape)
lang = [c for c in df.columns if c.startswith("language.")]
print("  ", lang[0], "->", df[lang[0]].iloc[0][:80])
stats = json.load(open(f"{path}/meta/stats.json"))
print("stats:", {k: len(v["mean"]) for k, v in stats.items() if isinstance(v, dict) and "mean" in v})
print("✅ Dataset nạp được.")
EOF


In [ ]:
# [BƯỚC 8] Fine-tune chạy nền, log ghi ra $OUTPUT_DIR/train.log. Cell này kết thúc ngay sau khi khởi động;
# xem tiến độ và ETA bằng Bước 8b. Trình duyệt mất kết nối không làm dừng train (miễn runtime còn sống).
# Chỉ lưu trọng số model (--save-only-model) để checkpoint gọn; không tiếp tục train được từ checkpoint này.
import os, subprocess
os.makedirs(os.environ["OUTPUT_DIR"], exist_ok=True)
TRAIN_LOG = f"{os.environ['OUTPUT_DIR']}/train.log"
assert "train_proc" not in globals() or train_proc.poll() is not None, "Train đang chạy; xem Bước 8b."
train_cmd = f"""
cd /content/Isaac-GR00T && source .venv/bin/activate && bash examples/finetune.sh \\
    --base-model-path "$BASE_MODEL_PATH" \\
    --dataset-path "$DATASET_DIR" \\
    --embodiment-tag "$EMBODIMENT_TAG" \\
    --output-dir "$OUTPUT_DIR" \\
    --experiment-name "$EXPERIMENT" \\
    --save-only-model \\
    -- --gradient_accumulation_steps "$GRAD_ACC" > "{TRAIN_LOG}" 2>&1
"""
train_proc = subprocess.Popen(["bash", "-c", train_cmd], env={**os.environ, "PYTHONUNBUFFERED": "1"},
                              start_new_session=True)
print(f"Đã khởi động train (PID {train_proc.pid}). Log: {TRAIN_LOG}. Chạy ngay Bước 8a (chép checkpoint về Drive), "
      "rồi Bước 8b để xem tiến độ.")

In [ ]:
# [BƯỚC 8a] Chép từng checkpoint về Drive ngay khi ghi xong (luồng chạy nền, kiểm tra mỗi phút). Chạy ngay sau Bước 8,
# trước Bước 8c (Bước 8c chờ tới khi train xong).
# finetune.sh chỉ giữ 5 checkpoint gần nhất và ổ /content mất khi runtime bị ngắt, nên không chép thì checkpoint giữa
# chừng (để chọn checkpoint tốt nhất ở Bước 11) sẽ mất. Chép vào thư mục tạm "_partial_checkpoint-N" rồi đổi tên, nên
# thư mục "checkpoint-N" trên Drive luôn đầy đủ. Luồng tự dừng sau lần chép cuối khi train xong; runtime khởi động lại
# thì chạy lại cell này để chép nốt.
import glob, os, re, shutil, threading, time

SYNC_FROM_STEP = int(os.environ["SYNC_FROM_STEP"])
SRC = f"{os.environ['OUTPUT_DIR']}/{os.environ['EXPERIMENT']}"
DST = f"{os.environ['DRIVE_CKPT_DIR']}/{os.environ['EXPERIMENT']}"
step_of = lambda p: int(re.search(r"checkpoint-(\d+)$", p).group(1))

def dir_size(path):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(path) for f in fs)

def ready(path):
    # Trainer ghi trainer_state.json sau cùng; đợi thêm 60 s không có file nào thay đổi
    files = [os.path.join(r, f) for r, _, fs in os.walk(path) for f in fs]
    return (os.path.exists(f"{path}/trainer_state.json")
            and time.time() - max(os.path.getmtime(f) for f in files) > 60)

def sync_once(log):
    os.makedirs(DST, exist_ok=True)
    for src in sorted(glob.glob(f"{SRC}/checkpoint-*"), key=step_of):
        step = step_of(src)
        dest = f"{DST}/checkpoint-{step}"
        if step < SYNC_FROM_STEP or os.path.isdir(dest) or not ready(src):
            continue
        size = dir_size(src)
        free = shutil.disk_usage(DST).free
        if free < size * 1.1:
            log(f"⚠️ Drive còn {free / 1e9:.1f} GB, không đủ chép checkpoint-{step} ({size / 1e9:.1f} GB)")
            continue
        tmp = f"{DST}/_partial_checkpoint-{step}"
        shutil.rmtree(tmp, ignore_errors=True)
        t0 = time.time()
        shutil.copytree(src, tmp)
        os.rename(tmp, dest)
        log(f"✅ checkpoint-{step} -> Drive ({size / 1e9:.1f} GB, {time.time() - t0:.0f} s)")

sync_log = []
def sync_loop():
    log = lambda m: sync_log.append(f"[{time.strftime('%H:%M:%S')}] {m}")
    while True:
        training = "train_proc" in globals() and train_proc.poll() is None
        if not training:
            time.sleep(65)  # checkpoint cuối vừa ghi: đợi qua ngưỡng 60 s của ready() rồi mới chép lượt cuối
        try:
            sync_once(log)
        except Exception as e:  # Drive tạm lỗi: thử lại ở vòng sau
            log(f"❌ {type(e).__name__}: {e}")
            if not training:
                continue
        if not training:
            log("Train đã dừng, chép xong lượt cuối.")
            return
        time.sleep(60)

if "sync_thread" in globals() and sync_thread.is_alive():
    print("Luồng chép đang chạy.")
else:
    sync_thread = threading.Thread(target=sync_loop, daemon=True)
    sync_thread.start()
    print(f"Đã bật luồng chép checkpoint từ bước {SYNC_FROM_STEP}: {SRC} -> {DST}")
print("Nhật ký (chạy lại cell này để xem):", *sync_log[-10:], sep="\n  ")
print("Trên Drive:", sorted((os.path.basename(p) for p in glob.glob(f"{DST}/checkpoint-*")), key=lambda n: int(n.split("-")[1])))

In [ ]:
# [BƯỚC 8b] Theo dõi train: chạy lại cell này bất cứ lúc nào để xem bước hiện tại, ETA, loss và VRAM.
import os, re, subprocess
log = open(f"{os.environ['OUTPUT_DIR']}/train.log", errors="replace").read()
# Thanh tiến độ của HuggingFace Trainer dạng "150/5000 [05:10<2:47:30,  2.07s/it]"
bars = re.findall(r"(\d+)/(\d+) \[([\d:]+)<([\d:?]+),\s*([\d.]+)(s/it|it/s)", log)
bars = [b for b in bars if b[1] == os.environ["MAX_STEPS"]]
if bars:
    done, total, elapsed, eta, rate, unit = bars[-1]
    print(f"Bước {done}/{total} ({int(done) / int(total):.0%}) | đã chạy {elapsed} | còn lại ~{eta} | {rate} {unit}")
else:
    print("Chưa vào vòng train (đang tải model / dựng dữ liệu). 20 dòng log cuối:")
    print("\n".join(log.replace("\r", "\n").splitlines()[-20:]))
losses = re.findall(r"'loss': ([\d.e-]+)", log)
if losses:
    print(f"Loss gần nhất: {losses[-1]} (đầu tiên: {losses[0]}, {len(losses)} lần ghi, mỗi 10 bước)")
if "train_proc" not in globals():
    status = None
    print("Trạng thái: không rõ (kernel đã khởi động lại nên mất tiến trình train; xem log ở trên)")
else:
    status = train_proc.poll()
    print("Trạng thái:", "đang chạy" if status is None else f"đã kết thúc, mã thoát {status}")
if status not in (None, 0):
    print("Train lỗi. 30 dòng log cuối:")
    print("\n".join(log.replace("\r", "\n").splitlines()[-30:]))
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu", "--format=csv"],
                     capture_output=True, text=True).stdout)

In [ ]:
# [BƯỚC 8c] Chờ tới khi train xong: in tiến độ mỗi phút, xong thì phát tiếng bíp.
# Dừng cell này (■) lúc nào cũng được, train vẫn chạy tiếp.
import os, re, time
import numpy as np
from IPython.display import Audio, clear_output, display

while train_proc.poll() is None:
    log = open(f"{os.environ['OUTPUT_DIR']}/train.log", errors="replace").read()
    bars = [b for b in re.findall(r"(\d+)/(\d+) \[([\d:]+)<([\d:?]+)", log) if b[1] == os.environ["MAX_STEPS"]]
    losses = re.findall(r"'loss': ([\d.e-]+)", log)
    clear_output(wait=True)
    print(time.strftime("%H:%M:%S"), "|",
          f"bước {bars[-1][0]}/{bars[-1][1]}, còn lại ~{bars[-1][3]}" if bars else "đang khởi động",
          f"| loss {losses[-1]}" if losses else "")
    time.sleep(60)

code = train_proc.returncode
print("✅ Train xong." if code == 0 else f"❌ Train dừng với mã thoát {code}: chạy Bước 8b để xem log lỗi.")
t = np.linspace(0, 1.5, 24000)
display(Audio(np.sin(2 * np.pi * 880 * t) * (t % 0.5 < 0.25), rate=16000, autoplay=True))

In [ ]:
# [BƯỚC 8d] Biểu đồ loss, grad norm và learning rate theo bước train. Chạy được trong lúc train (đọc train.log)
# hoặc sau khi runtime khởi động lại (đọc trainer_state.json trong checkpoint trên ổ tạm hoặc Drive).
# Ảnh lưu ở $OUTPUT_DIR/loss_curve.png và cạnh checkpoint trên Drive.
import ast, glob, json, os, re, shutil
import numpy as np
import matplotlib.pyplot as plt

def from_train_log(path):
    # HF Trainer in mỗi lần ghi log dạng {'loss': ..., 'grad_norm': ..., 'learning_rate': ..., 'epoch': ...};
    # bước của lần ghi lấy từ thanh tiến độ "N/MAX_STEPS [" gần nhất phía trước nó
    if not os.path.exists(path):
        return []
    log = open(path, errors="replace").read()
    bars = [(m.start(), int(m.group(1))) for m in re.finditer(r"(\d+)/" + os.environ["MAX_STEPS"] + r" \[", log)]
    rows = []
    for m in re.finditer(r"\{[^{}]*'loss'[^{}]*\}", log):
        try:
            d = {k: float(v) for k, v in ast.literal_eval(m.group(0)).items()}
        except (ValueError, SyntaxError, TypeError):
            continue
        before = [s for pos, s in bars if pos < m.start()]
        d["step"] = before[-1] if before else 10 * (len(rows) + 1)
        rows.append(d)
    return rows

def from_trainer_state():
    dirs = [f"{os.environ['OUTPUT_DIR']}/{os.environ['EXPERIMENT']}", f"{os.environ['DRIVE_CKPT_DIR']}/{os.environ['EXPERIMENT']}"]
    for d in dirs:
        states = sorted(glob.glob(f"{d}/checkpoint-*/trainer_state.json"),
                        key=lambda p: int(re.search(r"checkpoint-(\d+)", p).group(1)))
        if states:
            print("Đọc", states[-1])
            return [e for e in json.load(open(states[-1]))["log_history"] if "loss" in e]
    return []

rows = from_train_log(f"{os.environ['OUTPUT_DIR']}/train.log") or from_trainer_state()
assert rows, "Chưa có loss nào: train chưa tới lần ghi log đầu tiên (10 bước), hoặc không thấy train.log / checkpoint."

step = np.array([r["step"] for r in rows], dtype=float)
series = {k: np.array([r.get(k, np.nan) for r in rows], dtype=float) for k in ("loss", "grad_norm", "learning_rate")}
loss = series["loss"]
win = max(5, len(loss) // 20)  # trung bình trượt khoảng 5% số lần ghi
smooth = np.array([loss[max(0, i - win + 1):i + 1].mean() for i in range(len(loss))])

# Màu: cùng một hue xanh (nhạt = giá trị gốc, đậm = trung bình trượt); chữ và lưới dùng màu trung tính
BLUE, BLUE_LIGHT, INK, INK_2, GRID = "#2a78d6", "#86b6ef", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({"axes.edgecolor": GRID, "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2,
                     "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold", "font.size": 9})
fig, axes = plt.subplots(3, 1, figsize=(10, 8.5), sharex=True, gridspec_kw={"height_ratios": [2.2, 1, 1]})
for ax in axes:
    ax.grid(True, color=GRID, linewidth=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_facecolor("#fcfcfb")

ax = axes[0]
ax.plot(step, loss, color=BLUE_LIGHT, linewidth=1, label="loss (mỗi lần ghi log)")
ax.plot(step, smooth, color=BLUE, linewidth=2, label=f"trung bình trượt {win} lần ghi")
ax.set_yscale("log")
ax.set_title(f"Loss flow matching – {os.environ['EXPERIMENT']}", loc="left")
ax.set_ylabel("loss (thang log)")
ax.legend(frameon=False, loc="upper right", labelcolor=INK_2)
ax.annotate(f"{smooth[-1]:.4f}", (step[-1], smooth[-1]), xytext=(6, 0), textcoords="offset points",
            va="center", color=INK, fontweight="bold")

for ax, key, title in ((axes[1], "grad_norm", "Grad norm"), (axes[2], "learning_rate", "Learning rate")):
    vals = series[key]
    if np.isfinite(vals).any():
        ax.plot(step, vals, color=BLUE, linewidth=1.5)
    else:
        ax.text(0.5, 0.5, "không có trong log", transform=ax.transAxes, ha="center", color=INK_2)
    ax.set_title(title, loc="left")
axes[2].ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
axes[2].set_xlabel("bước train")
fig.tight_layout()

png = f"{os.environ['OUTPUT_DIR']}/loss_curve.png"
fig.savefig(png, dpi=130, facecolor="white")
plt.show()
drive_png = f"{os.environ['DRIVE_CKPT_DIR']}/{os.environ['EXPERIMENT']}/loss_curve.png"
os.makedirs(os.path.dirname(drive_png), exist_ok=True)
shutil.copy(png, drive_png)

i_min = int(np.nanargmin(smooth))
print(f"{len(loss)} lần ghi, bước {int(step[0])}–{int(step[-1])} | loss đầu {loss[0]:.4f} -> cuối (trung bình trượt) "
      f"{smooth[-1]:.4f}, giảm {loss[0] / smooth[-1]:.0f} lần | thấp nhất {smooth[i_min]:.4f} ở bước {int(step[i_min])}")
print("Đã lưu:", png, "và", drive_png)

In [ ]:
# [BƯỚC 9] Đánh giá open-loop trên tập val (episode không có trong train): so action model dự đoán với action trong
# dữ liệu. Log in MSE/MAE; biểu đồ lưu ở /tmp/open_loop_eval/traj_<id>.jpeg. Đánh giá thật sự (vòng kín, SONIC 1.1
# trong MuJoCo) chạy ở máy local: closed_loop_eval.py --gr00t HOST:PORT với server gr00t/eval/run_gr00t_server.py.
import glob, os, re
assert "train_proc" not in globals() or train_proc.poll() is not None, "Train chưa xong; xem tiến độ ở Bước 8b."
ckpts = sorted(glob.glob(f"{os.environ['OUTPUT_DIR']}/{os.environ['EXPERIMENT']}/checkpoint-*"),
               key=lambda p: int(re.search(r"checkpoint-(\d+)", p).group(1)))
assert ckpts, "Chưa có checkpoint: Bước 8 chưa chạy xong."
ckpt, dataset_dir = ckpts[-1], os.environ["TEST_DIR"]
os.environ["CKPT"] = ckpt  # Bước 10 dùng lại
print("Checkpoint:", ckpt)
# MPLBACKEND=Agg: Colab đặt backend matplotlib_inline, gói này không có trong môi trường Isaac-GR00T
!cd /content/Isaac-GR00T && MPLBACKEND=Agg .venv/bin/python gr00t/eval/open_loop_eval.py --model-path "{ckpt}" --dataset-path "{dataset_dir}" --embodiment-tag UNITREE_G1_SONIC --traj-ids 0 1 --steps 2000 --execution-horizon 15


In [ ]:
# [BƯỚC 10] Chép checkpoint về Google Drive để dùng với colab_gr00t_server.ipynb. Bước 8a đã chép mọi checkpoint từ
# SYNC_FROM_STEP; cell này để chép checkpoint trước mốc đó hoặc khi không bật Bước 8a.
# COPY_STEPS: "latest" (mới nhất), "all" (mọi checkpoint còn trên ổ tạm), hoặc danh sách bước, ví dụ [4000].
import glob, os, re, shutil
COPY_STEPS = "latest"
if "train_proc" in globals() and train_proc.poll() is None:
    print("⚠️ Train vẫn đang chạy: checkpoint mới nhất chưa phải checkpoint cuối.")
step_of = lambda p: int(re.search(r"checkpoint-(\d+)$", p).group(1))
ckpts = sorted(glob.glob(f"{os.environ['OUTPUT_DIR']}/{os.environ['EXPERIMENT']}/checkpoint-*"), key=step_of)
assert ckpts, f"Không thấy checkpoint trong {os.environ['OUTPUT_DIR']}/{os.environ['EXPERIMENT']}."
chosen = (ckpts[-1:] if COPY_STEPS == "latest" else ckpts if COPY_STEPS == "all"
          else [c for c in ckpts if step_of(c) in COPY_STEPS])
assert chosen, f"Không có checkpoint {COPY_STEPS} trên ổ tạm (còn: {[step_of(c) for c in ckpts]})."
for ckpt in chosen:
    dest = f"{os.environ['DRIVE_CKPT_DIR']}/{os.environ['EXPERIMENT']}/{os.path.basename(ckpt)}"
    if os.path.isdir(dest):
        print(f"Đã có trên Drive: {dest}")
        continue
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    print(f"Đang chép {ckpt} -> {dest} ...")
    tmp = f"{os.path.dirname(dest)}/_partial_{os.path.basename(ckpt)}"
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(ckpt, tmp)
    os.rename(tmp, dest)
    size_gb = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(dest) for f in fs) / 1e9
    print(f"✅ Đã chép ({size_gb:.1f} GB).")
print(f'Chạy server (Bước 5 của colab_gr00t_server.ipynb): MODEL_PATH = "{dest}", EMBODIMENT_TAG = "new_embodiment"')
print("Ở máy local thêm --vla-action-hz 25.")